<a href="https://colab.research.google.com/github/niofrequency/google-collab_image/blob/main/HEADSWAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 1. Install dependencies
# Only the model libraries are upgraded (Qwen-Image 2.1 needs new ones). The rest is installed only if it's missing,
# so Colab's own packages (requests, pillow, huggingface_hub, …) keep the versions Colab expects.
!pip install -q --upgrade "torchao>=0.16.0" "diffusers>=0.43.0" transformers accelerate peft
!pip install -q safetensors pillow opencv-python-headless fastapi uvicorn python-multipart huggingface_hub requests

# "Expression from the body photo" (website switch): LivePortrait's code; its models (~680 MB) download on first use
!test -d /content/LivePortrait || git clone -q https://github.com/KwaiVGI/LivePortrait /content/LivePortrait
!cd /content/LivePortrait && git checkout -q 9b294b3
!pip install -q onnx onnxruntime tyro rich scikit-image pyyaml imageio
# sharper enlarging of the result to the body photo's size (4x-UltraSharp; its model downloads on first use)
!pip install -q spandrel

from IPython.display import clear_output
clear_output()
print("✅ Installed")

In [ ]:
#@title 2. Download & verify Qwen-Image 2.1 + BFS Head Swap LoRA weights
#@markdown **Head swap LoRA** (both by the BFS author, for Qwen-Image 2.1):
#@markdown - `v1.1`: recommended. Best skin detail and head angle.
#@markdown - `v1.1 alternative`: copies the expression better (open mouth, strong emotions); head angle is a little less exact.
HEAD_LORA = "v1.1 (recommended)" #@param ["v1.1 (recommended)", "v1.1 alternative (stronger expressions)"]
import gc
import os
import torch
from huggingface_hub import hf_hub_download, snapshot_download

torch.cuda.empty_cache()
gc.collect()

LORA_DIR = "/content/loras"
os.makedirs(LORA_DIR, exist_ok=True)

LORA_FILE = {
    "v1.1 (recommended)": "bfs_head_v1.1_qwen_2.1.safetensors",
    "v1.1 alternative (stronger expressions)": "bfs_head_v1.1_alternative_qwen_2.1.safetensors",
}[HEAD_LORA]
os.environ["HEADSWAP_BASE_LORA"] = LORA_FILE  # read by the server started in cell 4

print("Downloading/verifying base model (Qwen/Qwen-Image-2.1)...")
snapshot_download(
    repo_id="Qwen/Qwen-Image-2.1",
    ignore_patterns=["*.msgpack", "*.h5", "*.ot"],
)

print(f"Downloading BFS Head Swap LoRA ({LORA_FILE})...")
lora_path = hf_hub_download(
    repo_id="Alissonerdx/BFS-Best-Face-Swap",
    filename=LORA_FILE,
    local_dir=LORA_DIR,
)

with open(lora_path, "rb") as f:
    header = f.read(16)
if header.startswith(b"{") or header.startswith(b"<"):
    raise RuntimeError(f"Downloaded file is not a valid safetensors binary: {lora_path}")

lora_size_mb = os.path.getsize(lora_path) / (1024 * 1024)
print(f"✅ Weights cached. LoRA path: {lora_path} ({lora_size_mb:.2f} MB)")
print("ℹ️ Don't add a body swap LoRA (bfs_body_swap_...) for head swaps: it is made for a different job and lowers quality.")

In [ ]:
#@title 3A. Clear kernel VRAM
import gc
import torch

if "pipe" in globals():
    del pipe

gc.collect()
torch.cuda.empty_cache()
print("✅ Notebook kernel GPU memory cleared.")

In [ ]:
%%writefile /content/headswap_server.py
# Head Swap backend: Qwen-Image 2.1 + BFS Head Swap LoRA, served for the website's Head Swap tab.
import os

# Must be set before torch is imported: lets PyTorch reuse fragmented GPU memory between requests.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import base64
import sys
import gc
import glob
import math
import random
import re
import threading
import time
import traceback
import urllib.parse
import urllib.request
import uuid
from collections import deque
from contextlib import asynccontextmanager
from io import BytesIO
from typing import Optional

import numpy as np
import requests
import torch
from diffusers import DiffusionPipeline
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image
from pydantic import BaseModel
from safetensors.torch import load_file

pipe: Optional[DiffusionPipeline] = None
LOG_BUFFER = deque(maxlen=200)
LORA_DIR = "/content/loras"
# The head swap LoRA, always on. Chosen in cell 2 (v1.1 = recommended, alternative = stronger expressions).
BASE_LORA = os.environ.get("HEADSWAP_BASE_LORA", "bfs_head_v1.1_qwen_2.1.safetensors")
BASE_ADAPTER = "bfs_head"
MODEL_EXTS = (".safetensors",)
CIVITAI_TOKEN = os.environ.get("CIVITAI_TOKEN", "")
PIPE_LOCK = threading.Lock()   # one swap at a time on the GPU
LOADED_ADAPTERS = {}           # LoRA file name -> adapter name
JOBS = {}

# Quality settings, matching the official Qwen-Image 2.1 pipeline (BFS guide: LoRA strength 1.0, cfg 1).
DEFAULT_STEPS = 40          # Qwen-Image 2.1 is made for 40-50 steps without a speed LoRA; 20 leaves results soft
DEFAULT_RESOLUTION = 1024   # output and reference images are ~1024x1024 pixels in area (the model's default)
MAX_RESOLUTION = 1536       # higher looks sharper but needs more memory; falls back automatically on the L4
MULTIPLE = 32               # Qwen-Image 2.1 sizes must be multiples of 32

# Fast mode: PrunaAI's 8-step speed LoRA for Qwen-Image 2.1 (the BFS author's reference workflow uses it together
# with the head swap LoRA). 8 steps instead of 40, no CFG, with the sigma schedule PrunaAI gives for it.
FAST_LORA = ("PrunaAI/P-Q-Image-2.1", "p_q_image_2.1_8_steps_v0.1.safetensors")
FAST_ADAPTER = "speed_8step"
FAST_SIGMAS = [1.0, 14 / 15, 6 / 7, 10 / 13, 2 / 3, 6 / 11, 0.4, 2 / 9]
SCHEDULERS = {}   # "base" (the pipeline's own) and "fast" (no extra shift, for the 8-step LoRA)

DEFAULT_PROMPT = (
    "head_swap: start with <image1> as the base image, keeping its lighting, environment, and background. "
    "remove the head from <image1> completely and replace it with the head from <image2>, strictly preserving "
    "the hair, eye color, nose structure from <image2>. copy the direction of the eye, head rotation, micro "
    "expressions from <image1>, high quality, sharp details, 4k"
)


def log_msg(msg: str) -> None:
    line = str(msg)
    LOG_BUFFER.append(line)
    print(line, flush=True)


def free_gpu() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def find_lora_file(lora_dir: str = LORA_DIR) -> str:
    # Always the HEAD swap LoRA. (Picking the first file alphabetically would load bfs_body_swap_... instead.)
    preferred = os.path.join(lora_dir, BASE_LORA)
    if os.path.exists(preferred):
        return preferred
    candidates = sorted(glob.glob(os.path.join(lora_dir, "bfs_head*.safetensors")))
    if not candidates:
        raise FileNotFoundError(f"The BFS head swap LoRA ({BASE_LORA}) is missing from {lora_dir}. Run cell 2 again.")
    return candidates[0]


def list_lora_files():
    return sorted(f for f in os.listdir(LORA_DIR) if f.lower().endswith(MODEL_EXTS)) if os.path.isdir(LORA_DIR) else []


def adapter_name_for(file_name: str) -> str:
    return "user_" + re.sub(r"[^0-9a-zA-Z_]", "_", os.path.splitext(file_name)[0])[:60]


def load_lora(pipeline, path: str, adapter_name: str) -> None:
    try:
        pipeline.load_lora_weights(path, adapter_name=adapter_name)
    except Exception as lora_err:
        log_msg(f"Standard load_lora_weights failed for {os.path.basename(path)} ({lora_err}); retrying with remapped keys")
        try:
            pipeline.load_lora_weights(map_lora_state_dict(load_file(path)), adapter_name=adapter_name)
        except Exception as err:
            raise ValueError(
                f"LoRA '{os.path.basename(path)}' could not be loaded. It is probably not a Qwen-Image 2.1 LoRA. ({err})"
            ) from err


def map_lora_state_dict(state_dict: dict) -> dict:
    """Normalizes ComfyUI/Civitai LoRA prefixes while preserving Qwen-Image block names."""
    mapped = {}
    for k, v in state_dict.items():
        nk = k
        if nk.startswith("diffusion_model."):
            nk = nk[len("diffusion_model."):]
        if not nk.startswith("transformer.") and "transformer_blocks." in nk:
            nk = "transformer." + nk
        mapped[nk] = v
    return mapped


def load_pipeline() -> DiffusionPipeline:
    free_gpu()
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    log_msg(f"Loading Qwen-Image 2.1 in {dtype} ...")

    pipeline = DiffusionPipeline.from_pretrained("Qwen/Qwen-Image-2.1", torch_dtype=dtype)

    lora_file = find_lora_file(LORA_DIR)
    log_msg(f"Loading head swap LoRA: {lora_file}")
    load_lora(pipeline, lora_file, BASE_ADAPTER)
    LOADED_ADAPTERS.clear()
    LOADED_ADAPTERS[os.path.basename(lora_file)] = BASE_ADAPTER

    # The model is ~32 GB in bf16 (transformer ~14 GB + text encoder ~17.5 GB), more than an L4 (22 GB) holds.
    # Only put everything on the GPU when it clearly fits (A100 80 GB, H100); otherwise keep one part on the GPU
    # at a time. Never try a full .to("cuda") first: a failed move leaves memory behind and later swaps run out.
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    if total_gb >= 45:
        pipeline.to("cuda")
        log_msg(f"GPU has {total_gb:.0f} GB: whole model on the GPU.")
    else:
        pipeline.enable_model_cpu_offload()
        log_msg(f"GPU has {total_gb:.0f} GB: model CPU offload (one part on the GPU at a time).")

    # No VAE tiling: decoding a ~1 MP image in tiles leaves faint seams / pink-green grid lines on smooth skin,
    # and the whole image fits in memory anyway. The VAE runs in bf16 like the rest (not fp16, which tints skin).
    vae = getattr(pipeline, "vae", None)
    if vae is not None:
        if hasattr(vae, "disable_tiling"):
            vae.disable_tiling()

    # Re-rolls with the same pictures and prompt (another seed, other steps) reuse the encoded prompt + pictures,
    # so the ~17 GB text encoder doesn't have to be moved onto the GPU again.
    install_prompt_cache(pipeline)
    from diffusers import FlowMatchEulerDiscreteScheduler
    SCHEDULERS["base"] = pipeline.scheduler
    SCHEDULERS["fast"] = FlowMatchEulerDiscreteScheduler.from_config(
        pipeline.scheduler.config, use_dynamic_shifting=False, shift=1.0, shift_terminal=None)

    free_gpu()
    return pipeline


PROMPT_CACHE = {}   # key -> (prompt_embeds, prompt_embeds_mask, image_pad_mask); the last 2 are kept


def install_prompt_cache(pipeline):
    import hashlib
    original = pipeline.encode_prompt

    def encode_prompt(prompt, image=None, device=None, num_images_per_prompt=1, prompt_embeds=None,
                      prompt_embeds_mask=None, image_pad_mask=None):
        if prompt_embeds is not None or image is None:
            return original(prompt, image=image, device=device, num_images_per_prompt=num_images_per_prompt,
                            prompt_embeds=prompt_embeds, prompt_embeds_mask=prompt_embeds_mask, image_pad_mask=image_pad_mask)
        h = hashlib.sha1(repr((prompt, num_images_per_prompt)).encode())
        for img in image if isinstance(image, list) else [image]:
            h.update(repr(getattr(img, "size", "")).encode())
            h.update(img.tobytes() if hasattr(img, "tobytes") else repr(img).encode())
        key = h.hexdigest()
        if key in PROMPT_CACHE:
            log_msg("Reusing the encoded prompt and pictures (same pictures and prompt as before)")
            return PROMPT_CACHE[key]
        result = original(prompt, image=image, device=device, num_images_per_prompt=num_images_per_prompt)
        PROMPT_CACHE[key] = result
        while len(PROMPT_CACHE) > 2:
            PROMPT_CACHE.pop(next(iter(PROMPT_CACHE)))
        return result

    pipeline.encode_prompt = encode_prompt


@asynccontextmanager
async def lifespan(app: FastAPI):
    global pipe
    pipe = load_pipeline()
    log_msg("Model + LoRA loaded and ready")
    yield
    pipe = None
    free_gpu()


app = FastAPI(lifespan=lifespan)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


class LoraChoice(BaseModel):
    name: str
    strength: float = 1.0



# ---------------- Sharper enlarging ----------------
# The swap is made at ~1024 px and then brought back to the body photo's size. Plain resizing makes a big photo soft,
# so when it has to grow by more than ~15% it's enlarged with 4x-UltraSharp first (crisp skin, hair and lashes),
# then resized to the exact size. Its model (67 MB) downloads the first time; plain resizing is the fallback.
UPSCALER = ("4x-UltraSharp.pth", "https://huggingface.co/lokCX/4x-Ultrasharp/resolve/main/4x-UltraSharp.pth")
_upscaler = {}


def enlarge(img, size, job=None):
    if size[0] <= img.width * 1.15 and size[1] <= img.height * 1.15:
        return img.resize(size, Image.Resampling.LANCZOS)
    try:
        if "model" not in _upscaler:
            from spandrel import ModelLoader
            path = os.path.join("/content/upscale_models", UPSCALER[0])
            if not os.path.exists(path):
                if job is not None:
                    job.update(message="Downloading 4x-UltraSharp (once, 67 MB)")
                os.makedirs(os.path.dirname(path), exist_ok=True)
                urllib.request.urlretrieve(UPSCALER[1], path + ".part")
                os.replace(path + ".part", path)
            _upscaler["model"] = ModelLoader().load_from_file(path).eval()
        m = _upscaler["model"]
        if job is not None:
            job.update(message="Enlarging to the photo's size (4x-UltraSharp)")
        m.to("cuda").half()
        x = torch.from_numpy(np.array(img.convert("RGB"))).permute(2, 0, 1).float().div(255)[None].to("cuda").half()
        _, _, H, W = x.shape
        tile, pad, s = 512, 16, m.scale
        out = torch.zeros((1, 3, H * s, W * s), dtype=torch.float16, device="cpu")
        with torch.no_grad():
            for y in range(0, H, tile):
                for x0 in range(0, W, tile):
                    th, tw = min(tile, H - y), min(tile, W - x0)
                    ya, xa = max(0, y - pad), max(0, x0 - pad)
                    yb, xb = min(H, y + th + pad), min(W, x0 + tw + pad)
                    o = m(x[:, :, ya:yb, xa:xb])
                    out[:, :, y * s:(y + th) * s, x0 * s:(x0 + tw) * s] = \
                        o[:, :, (y - ya) * s:(y - ya + th) * s, (x0 - xa) * s:(x0 - xa + tw) * s].cpu()
        m.to("cpu").float()
        torch.cuda.empty_cache()
        big = Image.fromarray(out[0].float().clamp(0, 1).permute(1, 2, 0).mul(255).round().byte().numpy())
        return big.resize(size, Image.Resampling.LANCZOS)
    except Exception as e:   # never lose the swap over this
        log_msg(f"4x-UltraSharp not used ({e}) — plain resize instead")
        return img.resize(size, Image.Resampling.LANCZOS)



# ---------------- Expression from the target (LivePortrait) ----------------
# The head swap LoRA brings the whole head across — including the head photo's expression. This optional last
# step moves the BODY photo's facial expression (mouth, eyes, brows, gaze) onto the swapped face, keeping the new
# identity: LivePortrait (KwaiVGI, MIT code) reads both faces' 3D keypoints, takes the expression part from the
# target and the identity / head angle from the swap, re-renders the face and pastes it back with a soft mask.
# Its models (~680 MB) download the first time it's used; it runs in a second or two. The face detector it uses
# (InsightFace) is licensed for non-commercial use.
LP_DIR = "/content/LivePortrait"
LP_FILES = ["liveportrait/base_models/*", "liveportrait/retargeting_models/*", "liveportrait/landmark.onnx",
            "insightface/models/buffalo_l/*"]
LP_EXP = {   # which of LivePortrait's 21 expression keypoints (and which axes) belong to each region
    "mouth": [6, 12, 14, 17, 19, 20],
    "eyes": [11, 13, 15, 16, 18],
}
_lp = {}


def expression_available():
    return os.path.isdir(os.path.join(LP_DIR, "src"))


def load_liveportrait(job=None):
    if _lp:
        return _lp
    from huggingface_hub import snapshot_download
    if job is not None:
        job.update(message="Downloading LivePortrait for the expression (once, ~680 MB)")
    snapshot_download("KlingTeam/LivePortrait", local_dir=os.path.join(LP_DIR, "pretrained_weights"), allow_patterns=LP_FILES)
    if LP_DIR not in sys.path:
        sys.path.insert(0, LP_DIR)
    from src.config.inference_config import InferenceConfig
    from src.config.crop_config import CropConfig
    from src.live_portrait_wrapper import LivePortraitWrapper
    from src.utils.cropper import Cropper
    from src.utils.camera import get_rotation_matrix
    from src.utils.crop import prepare_paste_back, paste_back
    inf, crop_cfg = InferenceConfig(), CropConfig()
    _lp.update(wrapper=LivePortraitWrapper(inference_cfg=inf), cropper=Cropper(crop_cfg=crop_cfg), crop_cfg=crop_cfg,
               inf=inf, rotation=get_rotation_matrix, prepare_paste_back=prepare_paste_back, paste_back=paste_back)
    return _lp


def lp_to(device):
    """Moves LivePortrait's networks (~0.5 GB) on / off the GPU: they stay on the CPU between swaps, so the head
    swap model always has the whole GPU."""
    w = _lp.get("wrapper")
    if w is None:
        return
    for v in list(vars(w).values()):
        for m in (v.values() if isinstance(v, dict) else [v]):
            if isinstance(m, torch.nn.Module):
                m.to(device)


def transfer_expression(result_img, target_img, strength=1.0, region="all", job=None):
    """result_img with target_img's facial expression (same size). Returns (image, note)."""
    L = load_liveportrait(job)
    lp_to("cuda")
    try:
        return _transfer_expression(L, result_img, target_img, strength, region)
    finally:
        lp_to("cpu")
        torch.cuda.empty_cache()


def _transfer_expression(L, result_img, target_img, strength, region):
    w = L["wrapper"]
    src = np.array(result_img.convert("RGB"))
    drv = np.array(target_img.convert("RGB").resize(result_img.size, Image.Resampling.LANCZOS))
    crop_s = L["cropper"].crop_source_image(src, L["crop_cfg"])
    crop_d = L["cropper"].crop_source_image(drv, L["crop_cfg"])
    if crop_s is None or crop_d is None:
        return result_img, "no face found for the expression step — kept the swap as it is"
    with torch.no_grad():
        I_s = w.prepare_source(crop_s["img_crop_256x256"])
        s_info = w.get_kp_info(I_s)
        R_s = L["rotation"](s_info["pitch"], s_info["yaw"], s_info["roll"])
        f_s = w.extract_feature_3d(I_s)
        x_s = w.transform_keypoint(s_info)
        d_exp = w.get_kp_info(w.prepare_source(crop_d["img_crop_256x256"]))["exp"]
        exp = s_info["exp"].clone()
        if region in LP_EXP:
            for i in LP_EXP[region]:
                exp[:, i, :] = d_exp[:, i, :]
        else:   # the whole expression, exactly as LivePortrait's "expression" mode does it
            for i in [1, 2, 6, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]:
                exp[:, i, :] = d_exp[:, i, :]
            exp[:, 3:5, 1] = d_exp[:, 3:5, 1]
            exp[:, 5, 2] = d_exp[:, 5, 2]
            exp[:, 8, 2] = d_exp[:, 8, 2]
            exp[:, 9, 1:] = d_exp[:, 9, 1:]
        t = s_info["t"].clone()
        t[..., 2] = 0
        x_new = s_info["scale"] * (s_info["kp"] @ R_s + exp) + t   # the swap's identity and head angle, new expression
        x_new = w.stitching(x_s, x_new)                             # keeps the face edge lined up with the image
        x_new = x_s + (x_new - x_s) * float(strength)
        face = w.parse_output(w.warp_decode(f_s, x_s, x_new)["out"])[0]
    mask = L["prepare_paste_back"](L["inf"].mask_crop, crop_s["M_c2o"], dsize=(src.shape[1], src.shape[0]))
    out = L["paste_back"](face, crop_s["M_c2o"], src, mask).astype(np.float32)
    mask = mask if mask.ndim == 3 else mask[..., None]
    # LivePortrait draws the face at 512x512, which is softer than the swap on a big face. Put the swap's own fine
    # detail (pores, lashes, hair strands: everything finer than a few pixels) back on top of the new expression.
    import cv2
    srcf = src.astype(np.float32)
    face_px = max(1.0, float(np.sqrt(mask[..., 0].sum())))           # ~ the face's size in pixels
    sigma = max(1.0, min(6.0, face_px / 256.0))                       # how much softer the 512 px face is here
    detail = srcf - cv2.GaussianBlur(srcf, (0, 0), sigma)
    lowpass = cv2.GaussianBlur(out, (0, 0), sigma)
    out = mask * (lowpass + detail) + (1 - mask) * srcf
    torch.cuda.empty_cache()
    return Image.fromarray(np.clip(out, 0, 255).astype(np.uint8)), None


class HeadSwapRequest(BaseModel):
    body: str
    head: str
    prompt: str = ""
    num_inference_steps: Optional[int] = None
    resolution: Optional[int] = None
    seed: Optional[int] = None
    loras: list[LoraChoice] = []   # extra LoRAs on top of the head swap LoRA
    true_cfg_scale: Optional[float] = None   # 1 = off (official). Above 1 it uses negative_prompt; each step takes ~2x as long
    negative_prompt: str = ""
    fast: bool = False   # 8-step speed LoRA: ~5x faster, a little less fine detail
    expression_from_target: bool = False   # LivePortrait: copy the body photo's facial expression onto the new face
    expression_strength: float = 1.0       # 0.5 halfway ... 1 the target's expression ... 1.3 a bit stronger
    expression_region: str = "all"         # "all" (mouth, eyes, brows), "mouth" or "eyes"


class ModelDownload(BaseModel):
    url: str
    type: str = "lora"


def dataurl_to_pil(dataurl: str) -> Image.Image:
    """RGB image. Transparent parts (e.g. rounded screenshot corners) become white, not black: a black edge is
    copied into the result as a dark bar."""
    if "," in dataurl:
        dataurl = dataurl.split(",", 1)[1]
    img = Image.open(BytesIO(base64.b64decode(dataurl)))
    if img.mode in ("RGBA", "LA", "P"):
        img = img.convert("RGBA")
        bg = Image.new("RGB", img.size, (255, 255, 255))
        bg.paste(img, mask=img.split()[3])
        return bg
    return img.convert("RGB")


def pil_to_dataurl(img: Image.Image) -> str:
    buffered = BytesIO()
    img.convert("RGB").save(buffered, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buffered.getvalue()).decode("utf-8")


def output_size(img: Image.Image, resolution: int) -> tuple:
    """Body image's aspect ratio at about resolution x resolution pixels, in multiples of 32."""
    w, h = img.size
    ratio = w / h
    out_w = math.sqrt(resolution * resolution * ratio)
    out_h = out_w / ratio
    out_w = max(MULTIPLE, int(round(out_w / MULTIPLE)) * MULTIPLE)
    out_h = max(MULTIPLE, int(round(out_h / MULTIPLE)) * MULTIPLE)
    return out_w, out_h


def new_job(kind, **extra):
    job_id = uuid.uuid4().hex[:12]
    JOBS[job_id] = {"kind": kind, "status": "queued", "created": time.time(), "error": None, **extra}
    now = time.time()
    for jid in [j for j, v in JOBS.items() if v["status"] in ("done", "error") and now - v["created"] > 1800]:
        JOBS.pop(jid, None)
    return job_id


@app.get("/health")
def health():
    return {"status": "ok", "loaded": pipe is not None}


@app.get("/logs")
def get_logs():
    return {"logs": list(LOG_BUFFER), "lines": list(LOG_BUFFER)}


@app.get("/user_models")
def user_models():
    return {
        "model_type": "qwen_image_21_headswap",
        # Other BFS head swap files are left out: two head swap LoRAs at once fight each other
        "lora_files": [f for f in list_lora_files() if not f.startswith("bfs_head")],
        "base_lora": BASE_LORA,
        "supports_loras": True,
        # Settings the website can change (this model has one sampler and always makes a new image: no denoise)
        "controls": {"steps": [8, 60, DEFAULT_STEPS], "resolution": [512, MAX_RESOLUTION, DEFAULT_RESOLUTION],
                     "cfg": [1.0, 10.0, 1.0], "seed": True, "fast": True},
        "supports_expression": expression_available(),
    }


@app.get("/job/{job_id}")
def get_job(job_id: str):
    job = JOBS.get(job_id)
    if not job:
        return {"status": "error", "error": "Unknown job (the server restarted). Try again."}
    return {**job, "job_id": job_id}


# ---------------- LoRA downloads (Civitai / Hugging Face links) ----------------
def resolve_url(url):
    """Turn page links into direct download links. Returns (download_url, suggested_name)."""
    url = url.strip()
    parsed = urllib.parse.urlparse(url)
    qs = urllib.parse.parse_qs(parsed.query)
    token = (qs.get("token") or [CIVITAI_TOKEN])[0]
    host = parsed.netloc.lower()

    if "civitai.com" in host or "civitai.red" in host:
        api_headers = {"Authorization": f"Bearer {token}"} if token else {}
        m = re.search(r"/api/download/models/(\d+)", parsed.path)
        if m:
            version_id = m.group(1)
        else:
            version_id = (qs.get("modelVersionId") or [None])[0]
            if not version_id:
                mm = re.search(r"/models/(\d+)", parsed.path)
                if not mm:
                    raise ValueError("Unrecognised Civitai link. Use a model page or download link.")
                info = requests.get(f"https://civitai.com/api/v1/models/{mm.group(1)}", headers=api_headers, timeout=30)
                info.raise_for_status()
                versions = info.json().get("modelVersions") or []
                if not versions:
                    raise ValueError("That Civitai model has no downloadable versions.")
                version_id = versions[0]["id"]  # newest version
        url = f"https://civitai.com/api/download/models/{version_id}"
        if token:
            url += "?token=" + urllib.parse.quote(token)
        return url, None

    if "huggingface.co" in host:
        url = url.replace("/blob/", "/resolve/")
        return url, os.path.basename(urllib.parse.urlparse(url).path)

    return url, os.path.basename(parsed.path)


def filename_from_response(resp, fallback):
    cd = resp.headers.get("content-disposition", "")
    m = re.search(r"filename\*=(?:UTF-8'')?([^;]+)", cd, re.I) or re.search(r'filename="?([^";]+)"?', cd, re.I)
    name = urllib.parse.unquote(m.group(1).strip()) if m else (fallback or "")
    name = os.path.basename(name).strip().strip('"')
    return re.sub(r"[^\w.\-+() ]", "_", name)


def run_download(job_id, url):
    job = JOBS[job_id]
    job["status"] = "running"
    tmp_path = None
    try:
        os.makedirs(LORA_DIR, exist_ok=True)
        dl_url, suggested = resolve_url(url)
        with requests.get(dl_url, stream=True, timeout=60, allow_redirects=True,
                          headers={"User-Agent": "Mozilla/5.0"}) as r:
            if r.status_code in (401, 403):
                raise ValueError("Access denied: this model needs a Civitai API key")
            r.raise_for_status()
            if "text/html" in r.headers.get("content-type", ""):
                raise ValueError("The link returned a web page, not a model file. Use the download link.")
            name = filename_from_response(r, suggested)
            if not name.lower().endswith(MODEL_EXTS):
                name = f"lora_{int(time.time())}.safetensors"
            job["filename"] = name
            path = os.path.join(LORA_DIR, name)
            if os.path.exists(path):
                job["status"], job["progress"] = "done", 100
                return
            total = int(r.headers.get("content-length") or 0)
            job["total_mb"] = round(total / 1e6) if total else None
            tmp_path = path + ".part"
            done = 0
            with open(tmp_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                    f.write(chunk)
                    done += len(chunk)
                    job["done_mb"] = round(done / 1e6)
                    job["progress"] = round(done * 100 / total) if total else None
            os.replace(tmp_path, path)
            tmp_path = None
        job["status"], job["progress"] = "done", 100
        log_msg(f"Downloaded LoRA: {name} ({job.get('done_mb')} MB)")
    except Exception as e:
        log_msg(traceback.format_exc())
        job["status"], job["error"] = "error", str(e)
    finally:
        if tmp_path and os.path.exists(tmp_path):
            os.remove(tmp_path)


@app.post("/download_model")
def download_model(req: ModelDownload):
    job_id = new_job("download", type="lora", filename=None, progress=0, done_mb=0, total_mb=None)
    threading.Thread(target=run_download, args=(job_id, req.url), daemon=True).start()
    return {"job_id": job_id, "status": "queued", "type": "lora"}


def load_fast_lora(job=None):
    """Downloads (first time) and loads PrunaAI's 8-step speed LoRA as its own adapter."""
    if FAST_ADAPTER in LOADED_ADAPTERS.values():
        return
    if job is not None:
        job.update(message="Loading the 8-step speed LoRA (first time only)")
    from huggingface_hub import hf_hub_download
    path = hf_hub_download(FAST_LORA[0], FAST_LORA[1])
    log_msg(f"Loading the speed LoRA: {FAST_LORA[1]}")
    load_lora(pipe, path, FAST_ADAPTER)
    LOADED_ADAPTERS[FAST_LORA[1]] = FAST_ADAPTER


def apply_loras(choices, fast=False) -> str:
    """Turn on the head swap LoRA plus the chosen extra LoRAs (plus the speed LoRA in fast mode); all other
    loaded LoRAs are switched off."""
    names, weights = [BASE_ADAPTER], [1.0]
    if fast:
        names.append(FAST_ADAPTER)
        weights.append(1.0)
    for choice in choices:
        file_name = os.path.basename(choice.name)
        if file_name == BASE_LORA:
            weights[0] = float(choice.strength)
            continue
        path = os.path.join(LORA_DIR, file_name)
        if not os.path.exists(path):
            raise ValueError(f"LoRA '{file_name}' is not on the server. Download it again.")
        if file_name not in LOADED_ADAPTERS:
            log_msg(f"Loading LoRA {file_name} ...")
            load_lora(pipe, path, adapter_name_for(file_name))
            LOADED_ADAPTERS[file_name] = adapter_name_for(file_name)
        names.append(LOADED_ADAPTERS[file_name])
        weights.append(float(choice.strength))
    pipe.set_adapters(names, adapter_weights=weights)
    return ", ".join(f"{n}@{w:g}" for n, w in zip(names, weights))


def run_pipe(job, body_img, head_img, prompt, steps, resolution, use_kv_cache, seed, cfg=1.0, negative="", fast=False):
    width, height = output_size(body_img, resolution)
    log_msg(f"Running head swap: {width}x{height}, steps={steps}, cfg={cfg}, kv_cache={use_kv_cache}, seed={seed}")
    job.update(total_steps=steps, step=0, message=f"Generating {width}x{height}")

    def on_step_end(_pipe, step_index, _timestep, callback_kwargs):
        job.update(step=step_index + 1, progress=round(5 + 90 * (step_index + 1) / steps))
        log_msg(f"Step {step_index + 1}/{steps}")
        return callback_kwargs

    # no_grad, NOT inference_mode: with CPU offload, inference_mode turns the LoRA weights into "inference
    # tensors", and the next LoRA switch then fails with "Setting requires_grad=True on inference tensor
    # outside InferenceMode is not allowed".
    with torch.no_grad():
        return pipe(
            prompt=prompt,
            image=[body_img, head_img],   # <image1> = body (kept), <image2> = head (pasted in) - keep this order
            width=width,                 # always pass the size: otherwise the pipeline uses the LAST image (the head)
            height=height,
            output_resolution=resolution,  # the pipeline also resizes both reference images to this area
            num_inference_steps=steps,
            sigmas=FAST_SIGMAS if fast else None,
            true_cfg_scale=cfg,
            negative_prompt=negative if cfg > 1 else None,
            use_kv_cache=use_kv_cache,
            generator=torch.Generator("cpu").manual_seed(seed),
            callback_on_step_end=on_step_end,
        ).images[0]


# ---------------- Memory guard ----------------
# Colab kills the server when the runtime's RAM (12.7 GB on a T4) runs out. When free RAM gets low, this frees the
# caches this server keeps for speed (they're rebuilt when needed) and hands freed memory back to the system.
RAM_LOW_GB = 3.0        # before a job
RAM_CRITICAL_GB = 2.0   # after a job


def ram_available_gb():
    try:
        with open("/proc/meminfo") as fh:
            for line in fh:
                if line.startswith("MemAvailable:"):
                    return int(line.split()[1]) / 1024 / 1024
    except OSError:
        pass
    return 99.0


def memory_guard(after=False):
    limit = RAM_CRITICAL_GB if after else RAM_LOW_GB
    free = ram_available_gb()
    if free >= limit:
        return
    import ctypes
    import gc as _gc
    freed = []
    try:
        freed = free_caches() or []
    except Exception as e:
        print(f"Memory guard: could not free caches ({e})", flush=True)
    try:
        import comfy.model_management as _mm
        for fn in ("cleanup_models_gc", "cleanup_models"):
            try:
                getattr(_mm, fn)()
            except Exception:
                pass
    except Exception:
        pass
    _gc.collect()
    try:
        import torch as _torch
        _torch.cuda.empty_cache()
    except Exception:
        pass
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)   # give freed memory back to the system
    except Exception:
        pass
    print(f"Memory guard ({free:.1f} GB RAM free {'after' if after else 'before'} a job): freed "
          f"{', '.join(freed) or 'unused memory'} — {ram_available_gb():.1f} GB free now", flush=True)


def free_caches():
    freed = []
    if PROMPT_CACHE:
        PROMPT_CACHE.clear(); freed.append("cached prompts")
    extra = {f: a for f, a in LOADED_ADAPTERS.items() if a not in (BASE_ADAPTER, FAST_ADAPTER)}
    if extra:   # your extra LoRAs (the head swap and speed LoRAs stay)
        pipe.delete_adapters(list(extra.values()))
        for f in extra:
            LOADED_ADAPTERS.pop(f, None)
        freed.append(f"{len(extra)} extra LoRA(s)")
    return freed


def run_headswap(job_id, req: HeadSwapRequest):
    job = JOBS[job_id]
    job.update(status="queued", message="Waiting for the GPU (another swap is running)")
    with PIPE_LOCK:
        memory_guard()
        job.update(status="running", message="Preparing images", progress=3)
        try:
            body_img = dataurl_to_pil(req.body)
            head_img = dataurl_to_pil(req.head)
            prompt = req.prompt.strip() or DEFAULT_PROMPT
            steps = max(8, min(60, int(req.num_inference_steps or DEFAULT_STEPS)))
            resolution = max(512, min(MAX_RESOLUTION, int(req.resolution or DEFAULT_RESOLUTION)))
            seed = int(req.seed) if req.seed is not None and int(req.seed) >= 0 else random.randint(0, 2**31 - 1)
            cfg = max(1.0, min(10.0, float(req.true_cfg_scale or 1.0)))
            negative = (req.negative_prompt or "").strip() or "blurry, low quality, deformed face, extra head, seams, color streaks"
            fast = bool(req.fast)
            if fast:   # the speed LoRA is made for exactly 8 steps without CFG
                steps, cfg = len(FAST_SIGMAS), 1.0
                load_fast_lora(job)
            pipe.scheduler = SCHEDULERS["fast" if fast else "base"]
            t_start = time.time()

            job.update(message="Loading LoRAs")
            log_msg(f"LoRAs: {apply_loras(req.loras, fast)}")

            # Try the fast setting first, then lighter settings if the GPU runs out of memory.
            # The KV cache speeds up each step but holds ~3 GB (more at higher resolution).
            attempts = [(resolution, True), (resolution, False)]
            if resolution > DEFAULT_RESOLUTION:
                attempts.append((DEFAULT_RESOLUTION, False))
            if resolution > 768:
                attempts.append((768, False))

            result, last_error = None, None
            for res, kv in attempts:
                free_gpu()
                try:
                    result = run_pipe(job, body_img, head_img, prompt, steps, res, kv, seed, cfg, negative, fast)
                    break
                except torch.OutOfMemoryError as oom:
                    # Keep only the TEXT: the exception itself holds the failed attempt's GPU memory (~20 GB) until
                    # it's gone, so every lighter retry would start with a full GPU and fail too.
                    last_error = str(oom).split("\n")[0]
                    del oom
                    log_msg(f"Out of GPU memory at {res}px (kv_cache={kv}); trying a lighter setting...")
                    job.update(message="Out of GPU memory, retrying with a lighter setting")
            if result is None:
                raise RuntimeError(
                    "Out of GPU memory even at the lightest setting. Restart the Colab runtime and run all cells "
                    f"again, or use a bigger GPU (A100). Details: {last_error}"
                )

            # Back to the body photo's original size, so the result lines up with it
            if result.size != body_img.size:
                result = enlarge(result, body_img.size, job)
            expression_note = None
            if req.expression_from_target:
                if not expression_available():
                    expression_note = "the expression step isn't installed — open the notebook fresh from GitHub and run cell 1 again"
                else:
                    job.update(message="Copying the expression from the body photo", progress=97)
                    try:
                        result, expression_note = transfer_expression(
                            result, body_img, max(0.0, min(float(req.expression_strength or 1.0), 1.5)),
                            (req.expression_region or "all").lower(), job)
                        log_msg("Expression copied from the body photo" + (f" ({expression_note})" if expression_note else "") + ".")
                    except Exception as e:   # never lose the swap because of this optional step
                        traceback.print_exc()
                        expression_note = f"the expression step failed ({e}) — kept the swap as it is"

            seconds = round(time.time() - t_start, 1)
            job.update(status="done", progress=100, message="Done", image=pil_to_dataurl(result), seed=seed,
                       seconds=seconds, fast=fast, expression=bool(req.expression_from_target),
                       warnings=[f"Expression: {expression_note}"] if expression_note else [])
            log_msg(f"Done in {seconds} s ({'fast, 8 steps' if fast else f'{steps} steps'}).")
        except Exception as e:
            log_msg(traceback.format_exc())
            job.update(status="error", error=str(e) or type(e).__name__)
        finally:
            free_gpu()
            memory_guard(after=True)


@app.post("/headswap")
def headswap(req: HeadSwapRequest):
    if pipe is None:
        return {"error": "Pipeline not initialized"}
    # Runs in the background: a swap can take longer than the Cloudflare tunnel's ~100 s request limit.
    # The website polls /job/<id> for progress and the result.
    job_id = new_job("headswap", progress=0, step=0, total_steps=None, message="Queued", image=None)
    threading.Thread(target=run_headswap, args=(job_id, req), daemon=True).start()
    return {"job_id": job_id, "status": "queued"}


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="127.0.0.1", port=8000)

In [ ]:
#@title 4. Start server + Cloudflare tunnel
import os
import re
import subprocess
import sys
import time
import urllib.request

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn", "python-multipart"],
    check=True,
)

SERVER_SCRIPT = "/content/headswap_server.py"
SERVER_LOG = "/content/headswap_server.log"
TUNNEL_LOG = "/content/cloudflared.log"
RUNNER = "/content/run_headswap_server.sh"
PORT = 8000
CLOUDFLARED = "/usr/local/bin/cloudflared"

subprocess.run(["pkill", "-f", "run_headswap_server.sh"], capture_output=True)
subprocess.run(["pkill", "-f", "headswap_server.py"], capture_output=True)
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
time.sleep(1)

if not os.path.exists(CLOUDFLARED):
    print("Downloading cloudflared...")
    subprocess.run(
        [
            "wget",
            "-q",
            "-O",
            CLOUDFLARED,
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        ],
        check=True,
    )
    os.chmod(CLOUDFLARED, 0o755)

# The server runs in a loop: if it ever crashes, it starts again on the same port, so the
# website's Backend URL keeps working (after the model has loaded again, a few minutes).
with open(RUNNER, "w") as f:
    f.write(
        "#!/bin/bash\n"
        "while true; do\n"
        f"  {sys.executable} {SERVER_SCRIPT}\n"
        "  echo \"[runner] server stopped (exit code $?) - restarting in 5 s\"\n"
        "  sleep 5\n"
        "done\n"
    )
os.chmod(RUNNER, 0o755)

print("Starting Head Swap server...")
server_env = {**os.environ, "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
with open(SERVER_LOG, "w") as log:
    server_proc = subprocess.Popen(
        ["bash", RUNNER],
        env=server_env,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

print("Waiting for model to load and Uvicorn to bind port 8000...")
model_ready = False
for _ in range(450):  # loading the ~32 GB model can take several minutes
    time.sleep(2)
    if server_proc.poll() is not None:
        print("❌ Server process exited early. Check log below:")
        break
    if os.path.exists(SERVER_LOG):
        with open(SERVER_LOG) as f:
            content = f.read()
        if "Application startup complete" in content:
            try:
                with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=3) as resp:
                    if resp.status == 200 and b'"loaded":true' in resp.read().replace(b" ", b""):
                        print("✅ Model loaded and /health is ready")
                        model_ready = True
                        break
            except Exception:
                pass

if not model_ready:
    print("⚠️ Model did not finish loading in time. Last 30 log lines:")
    with open(SERVER_LOG) as f:
        print("".join(f.readlines()[-30:]))
    raise RuntimeError("HeadSwap server failed to become ready.")

print("Starting Cloudflare tunnel...")
# Its output goes to a file. (Reading it through a pipe that nobody empties later makes cloudflared
# freeze once the pipe is full, and then the website gets "Failed to fetch" / "Load failed".)
tunnel_log = open(TUNNEL_LOG, "w")
tunnel_proc = subprocess.Popen(
    [CLOUDFLARED, "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

TUNNEL_URL = None
t0 = time.time()
while time.time() - t0 < 90 and TUNNEL_URL is None:
    time.sleep(1)
    with open(TUNNEL_LOG) as f:
        m = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", f.read())
    if m:
        TUNNEL_URL = m.group(0)

if TUNNEL_URL:
    print("\n" + "=" * 60)
    print(f"READY: {TUNNEL_URL}")
    print("=" * 60)
    print("\n→ Copy the URL above and paste it into the Head Swap tab on your website.")
else:
    print("❌ Could not get Cloudflare URL. Try re-running this cell.")

In [ ]:
#@title 5. Show the latest server and tunnel logs (for troubleshooting)
!tail -40 /content/headswap_server.log
!echo; echo '--- tunnel ---'; tail -10 /content/cloudflared.log